# MLOps & Model Deployment

# Part 0:  Reusing NYC Yellow Taxi Trip Records dataset

## Data Cleaning

### Step 1: Data Ingestion and Storage

In [ ]:
from urllib.request import urlretrieve
from pathlib import Path

raw_dir = Path("data/raw")
raw_dir.mkdir(parents=True, exist_ok=True)

files = [
    ("https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet", raw_dir/"yellow_taxi_data.parquet"),
    ("https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv", raw_dir/"taxi_zone_lookup.csv"),
]

for url, filename in files:
    urlretrieve(url, filename)

print("Done!")

In [ ]:
import polars as pl

df = pl.read_parquet(raw_dir/'yellow_taxi_data.parquet')
df.head()

In [ ]:
import sys
expected_result = ["tpep_pickup_datetime", "tpep_dropoff_datetime", "PULocationID", "DOLocationID", "passenger_count", "trip_distance", "fare_amount", "tip_amount", "total_amount", "payment_type"]

try:
    missing = [m for m in expected_result if m not in df.columns]
    if missing:
        raise ValueError(f"Missing expected columns: {missing}")
    print("All columns present")

    if df.schema.get("tpep_pickup_datetime") != pl.Datetime:
        raise TypeError(f"tpep_pickup_datetime is {df.schema.get('tpep_pickup_datetime')}, expected Datetime")
    print("tpep_pickup_datetime datatype is: ", df.schema.get("tpep_pickup_datetime"))

    if df.schema.get("tpep_dropoff_datetime") != pl.Datetime:
        raise TypeError(f"tpep_dropoff_datetime is {df.schema.get('tpep_dropoff_datetime')}, expected Datetime")
    print("tpep_dropoff_datetime datatype is: ", df.schema.get("tpep_dropoff_datetime"))

    print(f'Number of rows: {len(df):,}')
    print(f'Number of columns: {len(df.columns)}')
    print('\nColumn names and types:')
    print(df.schema)

except Exception as e:
    print(f"Data Validation failed: {e}")
    sys.exit(1)


### Step 2: Data Transformation & Analysis

In [ ]:
# ensuring the data is clean to avoid issues in analysis and visualizations
# removing nulls values in all columns
df_clean = df.drop_nulls()
print(f'Number of rows after dropping nulls: {len(df_clean):,}')
print(f'Number of rows removed after dropping nulls: {len(df) - len(df_clean):,}')

#filtering invalid trips: trips with zero or negative distance, negative fares, or fares exceeding $500
df_invalid = (
    df_clean
    .filter((pl.col("fare_amount") > 0) & (pl.col("fare_amount") <= 500))
    .filter(pl.col("trip_distance") > 0)
)
print(f'Number of rows after filtering invalid trips: {len(df_invalid):,}')
print(f'Number of rows removed after filtering invalid trips: {len(df_clean) - len(df_invalid):,}')

#removing trips where dropoff time is before pickup time
df_final = df_invalid.filter(pl.col("tpep_dropoff_datetime") >= pl.col("tpep_pickup_datetime"))
print(f'Number of rows after filtering dropoff before pickup: {len(df_final):,}')
print(f'Number of rows removed after filtering dropoff before pickup: {len(df_invalid) - len(df_final):,}')


In [ ]:
#adding new column for trip duration in minutes
enriched = df_final.with_columns([
    ((pl.col("tpep_dropoff_datetime") - pl.col("tpep_pickup_datetime"))
    .dt.total_seconds() / 60).alias('trip_duration_minutes'),
])

enriched.select(pl.col("trip_duration_minutes")).head()

In [ ]:
#adding new column for trip speed
enriched = enriched.with_columns([
    (
        pl.when(pl.col("trip_duration_minutes") > 0)
        .then(pl.col("trip_distance") / (pl.col("trip_duration_minutes") / 60))
        .otherwise(None)
        .alias('trip_speed_mph')
    )
])

enriched.select(pl.col("trip_speed_mph"))

In [ ]:
#adding new columns for pickup hour and weekday
enriched = enriched.with_columns([
    pl.col('tpep_pickup_datetime').dt.hour().alias('pickup_hour'),
    pl.col('tpep_pickup_datetime').dt.weekday().alias('pickup_weekday')
])

enriched.select(pl.col("pickup_hour"), pl.col("pickup_weekday")).head()

In [ ]:
print(enriched.schema)

In [ ]:
print(enriched["payment_type"])

In [ ]:
# filtering the dataset to include only credit card transactions for the modeling tasks
enriched = enriched.filter(pl.col("payment_type").is_in([1]))


In [ ]:
print(enriched["payment_type"])

In [ ]:
print(len(enriched))    #dropped from 2.9 mil to 2.2 mil after filtering for credit card transactions

In [ ]:
enriched.write_parquet(raw_dir/'yellow_taxi_data_enriched.parquet')

## Data Preprocessing & Feature Engineering

### 1. Feature Engineering

In [ ]:
#temporal features

enriched = enriched.with_columns([
    pl.col('tpep_pickup_datetime').dt.hour().alias('pickup_hour'),
    (pl.col('tpep_pickup_datetime').dt.weekday() - 1).alias('pickup_day_of_week')
])

#days of the week 0 - 6 so weekend is 5 and 6
enriched = enriched.with_columns(
    (pl.col('pickup_day_of_week') >= 5).alias('is_weekend')
)

enriched.select(["tpep_pickup_datetime","pickup_hour","pickup_day_of_week","is_weekend"]).sample(n=5)

In [ ]:
#trip features

#trip duration minutes
enriched = enriched.with_columns([
    ((pl.col("tpep_dropoff_datetime") - pl.col("tpep_pickup_datetime"))
    .dt.total_seconds() / 60).alias('trip_duration_minutes')
])

#trip speed mph
enriched = enriched.with_columns([
    (
        pl.when(pl.col("trip_duration_minutes") > 0)
        .then(pl.col("trip_distance") / (pl.col("trip_duration_minutes") / 60))
        .otherwise(None)
        .alias('trip_speed_mph')
    )
])

#log trip distance
enriched = enriched.with_columns([
    pl.col("trip_distance").log1p().alias("log_trip_distance")
])

enriched.sample(n=5)

In [ ]:
# fare features
#fare_per_mile
enriched = enriched.with_columns([
    (
        pl.when(pl.col("trip_distance") > 0)
        .then (pl.col("fare_amount") / pl.col("trip_distance"))
        .otherwise(None)
        .alias('fare_per_mile')
    )
])

#fare_per_minute
enriched = enriched.with_columns([
    (
        pl.when(pl.col("trip_duration_minutes") > 0)
        .then(pl.col("fare_amount") / pl.col("trip_duration_minutes"))
        .otherwise(None)
        .alias('fare_per_minute')
    )
])

enriched.sample(n=5)

In [ ]:
df = pl.read_csv(raw_dir/'taxi_zone_lookup.csv')
df.head()

In [ ]:
#Zone features

# read zone table
zone_lookup = pl.read_csv(raw_dir/'taxi_zone_lookup.csv')

#join the zone lookup table to enriched table
enriched = enriched.join(
    zone_lookup.select([
        pl.col("LocationID"),
        pl.col("Borough")
    ]),
    left_on='PULocationID',
    right_on='LocationID',
    how='left'
).rename({'Borough': 'pickup_borough'})

enriched = enriched.join(
    zone_lookup.select([
        pl.col("LocationID"),
        pl.col("Borough")
    ]),
    left_on='DOLocationID',
    right_on='LocationID',
    how='left'
).rename({'Borough': 'dropoff_borough'})

enriched.sample(n=6)

In [ ]:
# convert from polars to pandas
import pandas as pd

enriched_pd = enriched.to_pandas()

enriched_pd.head()

In [ ]:
from sklearn.preprocessing import OneHotEncoder
#ensure a regular dense array is returned and sklearn do not crash with unknown category
ohe = OneHotEncoder(sparse_output=False, handle_unknown="ignore")

#look at columns and learn the categories
encoded_2d_array = ohe.fit_transform(enriched_pd[["pickup_borough", "dropoff_borough"]])

# to avoid the new columns being numbered 0,1,2...
encoded_cols = ohe.get_feature_names_out(["pickup_borough", "dropoff_borough"])

#turn encoded output to pandas table and add it to the enriched_pd dataframe
encoded_df = pd.DataFrame(encoded_2d_array, columns=encoded_cols, index=enriched_pd.index)
enriched_pd = pd.concat([enriched_pd, encoded_df], axis=1)

enriched_pd.sample(n=5)

### 2. Target Variable Creation

In [ ]:
# tip_amount (continuous, for regression)
y_reg = enriched_pd["tip_amount"]

#high_tip (binary: 1 if tip_amount > 20% of fare_amount, 0 otherwise, for classification
enriched_pd["high_tip"] = (
    (enriched_pd["tip_amount"] > 0) &
    (enriched_pd["tip_amount"] > 0.2 * enriched_pd["fare_amount"])
).astype(int)

y_clf = enriched_pd["high_tip"]

enriched_pd.sample(n=5)

In [ ]:

enriched.select(["tip_amount", "fare_amount"]).sample(n=5)
print(enriched_pd["tip_amount"].max())
print(enriched_pd["fare_amount"].max())

### 3. Data Splitting & Scaling

In [ ]:
#check max speed
print(f'Max speed: {enriched_pd["trip_speed_mph"].max()}')

In [ ]:
# Removing outliers
enriched_pd = enriched_pd[(enriched_pd["trip_speed_mph"] <= 100) & (enriched_pd["trip_speed_mph"] > 1)]
print(f'Max speed: {enriched_pd["trip_speed_mph"].max()}')
print(f'Min speed: {enriched_pd["trip_speed_mph"].min()}')

In [ ]:
#Dropping columns
columns_to_drop = [
    "RatecodeID",
    "trip_distance",
    "total_amount"
    ]

enriched_pd = enriched_pd.drop(columns_to_drop, axis=1)

In [ ]:
# Split data into training (70%), validation (15%), and test (15%) sets using stratified sampling for the classification target
from sklearn.model_selection import train_test_split

X= enriched_pd.drop(["tip_amount", "high_tip"], axis=1)
y_reg = enriched_pd["tip_amount"]
y_clf = enriched_pd["high_tip"]

X_train, X_temp, y_train_reg, y_temp_reg, y_train, y_temp = train_test_split(
    X, y_reg, y_clf, test_size=0.3, random_state=42, stratify=y_clf
)

X_val, X_test, y_val_reg, y_test_reg, y_val, y_test = train_test_split(
    X_temp, y_temp_reg, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

In [ ]:
#identify column types
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "bool"]).columns.tolist()

print(f'Numeric features: {len(numeric_features)}')
print(f'Categorical features: {len(categorical_features)}')

In [ ]:
# Apply StandardScaler to numeric features; fit on training data only
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

#numeric pipeline: impute missing values then scale
numeric_transformer = Pipeline(steps=[
  ('imputer', SimpleImputer(strategy='median')),
  ('scaler', StandardScaler())
])

preprocessor = ColumnTransformer(
transformers=[
  ('num', numeric_transformer, numeric_features),
])

X_train_scaled = preprocessor.fit_transform(X_train)
X_val_scaled = preprocessor.transform(X_val)
X_test_scaled = preprocessor.transform(X_test)


In [ ]:
included_features = numeric_features
excluded_by_preprocessor = [col for col in X.columns if col not in included_features]

print("Included in preprocessing:")
print(included_features)

print("\nExcluded by preprocessing:")
print(excluded_by_preprocessor)

In [ ]:
#Document the number of samples in each split and the class distribution of high_tip in each split
print("Number of samples in each split:")
print("Training set size:", len(X_train))
print("Validation set size:", len(X_val))
print("Test set size:", len(X_test))

print("\nClass distribution of high_tip in Training set:")
print(y_train.value_counts())
print(y_train.value_counts(normalize=True))

print("\nClass distribution of high_tip in Validation set:")
print(y_val.value_counts())
print(y_val.value_counts(normalize=True))

print("\nClass distribution of high_tip in Test set:")
print(y_test.value_counts())
print(y_test.value_counts(normalize=True))

#### Print a summary of feature names, types, and any features excluded from modeling (and why)

In [ ]:
print("Feature names used for modeling:")
print(numeric_features)
print("Number of features used:", len(numeric_features))

In [ ]:
print("\nFeature data types:")
print(X[numeric_features].dtypes)
# print(X.dtypes)

print("\nCount of each data type:")
print(X.dtypes.value_counts())

In [ ]:
print(f"\nNumeric features ({len(numeric_features)}):")
print(numeric_features)

print(f"\nCategorical features ({len(categorical_features)}):")
print(categorical_features)

In [ ]:
excluded_features = [col for col in X.columns if col not in numeric_features]

print("\nExcluded features:")
print(excluded_features)
print("Number of excluded features:", len(excluded_features))

In [ ]:
print("\nExcluded features:")
print("\ntip_amount was excluded because it is a regression target and would cause target leakage")
print("\nhigh_tip was excluded because it is a classification target and would cause target leakage")
print("\nVendorID was excluded it is an identifier and not included in the numeric preprocessing pipepline")
print("\ntpep_pickup_datetime was excluded because it is a date and not included in the numeric preprocessing pipepline")
print("\ntpep_dropoff_datetime was excluded because it is a date and not included in the numeric preprocessing pipepline")
print("\nstore_and_fwd_flag, is_weekend, pickup_borough and dropoff_borough was excluded because it is categorical")
print("\nPULocationID was excluded because it is a identifier")
print("\nDOLocationID was excluded because it is a identifier")
print("\npickup_hour, pickup_weekday and pickup_day_of_week was excluded because the data type is int8 and only data type int64 and float64 was included in the pipeline")

## Model Training 

### Baseline Models

In [ ]:
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)
import numpy as np
import pandas as pd

In [ ]:
#define regression models
reg_models = {
    "Linear Regression": LinearRegression(),
    "Random Forest Regressor": RandomForestRegressor(n_estimators=100, random_state=42)
   }


In [ ]:
trained_reg_pipeline = {}
reg_results = {}

for name, reg_model in reg_models.items():
    print(f"Training {name}...")
    pipe = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("regressor", reg_model)
    ])

    pipe.fit(X_train, y_train_reg)
    y_val_pred = pipe.predict(X_val)

    trained_reg_pipeline[name] = pipe

    reg_results[name] = {
        "MAE": mean_absolute_error(y_val_reg, y_val_pred),
        "RMSE": np.sqrt(mean_squared_error(y_val_reg, y_val_pred)),
        "R2": r2_score(y_val_reg, y_val_pred)
    }
print("Regression model training completed.")

In [ ]:
print("Regression Model Performance:")
for name, metrics in reg_results.items():
    print(f"{name}:")
    for metric_name, metric_value in metrics.items():
        print(f"  {metric_name}: {metric_value}")
    print()

In [ ]:
import os
import joblib

model_dir = "models"
os.makedirs(model_dir, exist_ok=True)

for name, pipe in trained_reg_pipeline.items():
    artifact={
        "model": pipe,
        "model_name": f"taxi-tip-{name.lower().replace(' ', '-')}",
        "feature_names": list(X_train.columns),
        "metrics": reg_results[name],
        "target_name": "tip_amount"
    }

    filename = f"{name.lower().replace(' ', '_')}_artifact.pkl"
    save_path = os.path.join(model_dir, filename)

    joblib.dump(artifact, save_path)
print(f"Regression model artifacts saved as {name} to {save_path}") 

AI USAGE
